Import Library & Setup

In [ ]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)

Load Raw Datasets (Simulasi Data Mentah)

In [ ]:
# Membuat simulasi data mentah Hotspot Karhutla (misal dari BMKG/SIPONGI)
np.random.seed(42)
date_range = pd.date_range(start="2023-01-01", end="2023-12-31", freq="D")

raw_hotspot = pd.DataFrame({
    'tanggal': date_range,
    'wilayah': np.random.choice(['Sumatera_Selatan', 'Riau', 'Kalimantan_Barat', 'Kalimantan_Tengah'], size=len(date_range)),
    'jumlah_hotspot': np.random.randint(0, 50, size=len(date_range))
})

# Sengaja menyuntikkan lonjakan hotspot di bulan Juli - Oktober untuk region tertentu
mask_spike = (raw_hotspot['tanggal'].dt.month.isin([7, 8, 9, 10])) & (raw_hotspot['wilayah'].isin(['Riau', 'Kalimantan_Tengah']))
raw_hotspot.loc[mask_spike, 'jumlah_hotspot'] = np.random.randint(100, 450, size=mask_spike.sum())

# Membuat simulasi data mentah Penjualan Masker (E-commerce)
raw_sales = pd.DataFrame({
    'order_date': date_range,
    'region': raw_hotspot['wilayah'],
    'units_sold': raw_hotspot['jumlah_hotspot'] * np.random.uniform(10, 15, size=len(date_range)) + np.random.normal(100, 25, size=len(date_range))
})
raw_sales['units_sold'] = raw_sales['units_sold'].astype(int)

print("--- RAW HOTSPOT DATA ---")
display(raw_hotspot.head())
print("\n--- RAW SALES DATA ---")
display(raw_sales.head())

Data Cleaning & Preprocessing (Hotspot Data)

In [ ]:
# 1. Salin dataframe agar data raw tetap aman
df_hotspot = raw_hotspot.copy()

# 2. Standarisasi nama kolom & tipe data
df_hotspot['tanggal'] = pd.to_datetime(df_hotspot['tanggal'])
df_hotspot['wilayah'] = df_hotspot['wilayah'].str.lower().str.strip()

# 3. Cek missing values dan duplikat
print("Missing values pada Hotspot:", df_hotspot.isnull().sum().to_dict())
print("Jumlah data duplikat:", df_hotspot.duplicated().sum())

# 4. Agregasi data menjadi mingguan (Weekly) untuk meredam noise harian
df_hotspot_weekly = df_hotspot.set_index('tanggal').groupby([
    pd.Grouper(freq='W'), 'wilayah'
])['jumlah_hotspot'].sum().reset_index()

print("\n--- Cleaned & Aggregated Hotspot Weekly ---")
display(df_hotspot_weekly.head())

Data Cleaning & Preprocessing (Sales Data)

In [ ]:
# 1. Salin dataframe
df_sales = raw_sales.copy()

# 2. Standarisasi format tanggal dan teks
df_sales['order_date'] = pd.to_datetime(df_sales['order_date'])
df_sales['region'] = df_sales['region'].str.lower().str.strip()

# 3. Handle anomaly/outliers (misal: jika ada nilai negatif akibat error sistem)
df_sales = df_sales[df_sales['units_sold'] >= 0]

# 4. Agregasi penjualan mingguan agar sejajar dengan data hotspot
df_sales_weekly = df_sales.set_index('order_date').groupby([
    pd.Grouper(freq='W'), 'region'
])['units_sold'].sum().reset_index()

print("--- Cleaned & Aggregated Sales Weekly ---")
display(df_sales_weekly.head())

Merging Datasets (Penggabungan Data)

In [ ]:
# Menggabungkan data hotspot dan penjualan berdasarkan Tanggal Minggu & Wilayah
df_merged = pd.merge(
    df_hotspot_weekly, 
    df_sales_weekly, 
    left_on=['tanggal', 'wilayah'], 
    right_on=['order_date', 'region'], 
    how='inner'
)

# Bersihkan kolom duplikat setelah merge
df_merged = df_merged.drop(columns=['order_date', 'region'])
df_merged = df_merged.rename(columns={'tanggal': 'week_date', 'jumlah_hotspot': 'total_hotspots', 'units_sold': 'total_mask_sales'})

print("--- Final Merged Dataset Shape:", df_merged.shape, "---")
display(df_merged.head(10))

Export Processed Data
Menyimpan hasil pembersihan ke dalam folder data/processed/ agar bisa langsung ditarik di notebook analisis berikutnya (02_eda_and_insights.ipynb).

In [ ]:
import os

# Membuat folder processed jika belum ada
os.makedirs('../data/processed', exist_ok=True)

# Export ke CSV
df_merged.to_csv('../data/processed/clean_karhutla_mask_sales.index = False', index=False)
print("Data bersih berhasil disimpan ke 'data/processed/clean_karhutla_mask_sales.csv'!")